# Fetch World Bank Projects

Paginates the WB Projects API, saves raw JSON pages, flattens to one row per project, and builds two yearly aggregates:
- `(sector, year)` totals → `wb_sector_year.parquet`
- `(country, year)` totals → `wb_country_year.parquet`
- Project-level rows → `wb_panel.parquet`

## 0. Setup

In [10]:
!pip install requests pandas pyarrow

In [11]:
import json
import logging
import time
from pathlib import Path

import pandas as pd
import requests

ROOT = Path.cwd().parent
RAW_DIR = ROOT / "data" / "raw" / "wb_projects"
INTERIM = ROOT / "data" / "interim"
RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM.mkdir(parents=True, exist_ok=True)

API = "https://search.worldbank.org/api/v2/projects"
FIELDS = (
    "id,project_name,countryshortname,countrycode,regionname,"
    "sector,sector1,sector2,theme,theme1,"
    "boardapprovaldate,closingdate,status,"
    "totalamt,grantamt,lendinginstr,lendprojectcost"
)
ROWS_PER_PAGE = 500
MAX_OFFSET = 35_000   # safe upper bound; API has ~28k projects
SLEEP = 0.5

logging.basicConfig(format="%(asctime)s %(levelname)s %(message)s",
                    level=logging.INFO, datefmt="%H:%M:%S")
log = logging.getLogger("wb")
print("ROOT:", ROOT)

ROOT: /Users/latahviawilliams/Downloads/Big_Data_export/final_project/ecosoc


## 1. Paginate the API

We save each page's raw JSON so reruns are cheap (skip-if-exists).

In [12]:
session = requests.Session()
session.headers.update({"Accept": "application/json",
                        "User-Agent": "ecosoc-research/0.1 (lwilliams127@uchicago.edu)"})

offset = 0
total = None
pages_fetched = 0
while offset < MAX_OFFSET:
    page_path = RAW_DIR / f"page_{offset:06d}.json"
    if page_path.exists() and page_path.stat().st_size > 100:
        log.info("skip cached %s", page_path.name)
    else:
        params = {"format": "json", "fl": FIELDS, "rows": ROWS_PER_PAGE, "os": offset}
        r = session.get(API, params=params, timeout=60)
        r.raise_for_status()
        page_path.write_bytes(r.content)
        log.info("fetched offset=%s bytes=%d", offset, len(r.content))
        time.sleep(SLEEP)

    payload = json.loads(page_path.read_text())
    if total is None:
        total = int(payload.get("total", 0))
        log.info("total projects reported: %s", total)
    projects = payload.get("projects", {})
    if not projects:
        log.info("empty page at offset=%s; stopping", offset)
        break
    pages_fetched += 1
    offset += ROWS_PER_PAGE
    if total and offset >= total:
        break
print(f"done. {pages_fetched} pages, total={total}")

17:27:36 INFO skip cached page_000000.json
17:27:36 INFO total projects reported: 22729
17:27:36 INFO skip cached page_000500.json
17:27:37 INFO skip cached page_001000.json
17:27:37 INFO skip cached page_001500.json
17:27:37 INFO skip cached page_002000.json
17:27:37 INFO skip cached page_002500.json
17:27:37 INFO skip cached page_003000.json
17:27:37 INFO skip cached page_003500.json
17:27:37 INFO skip cached page_004000.json
17:27:37 INFO skip cached page_004500.json
17:27:37 INFO skip cached page_005000.json
17:27:37 INFO skip cached page_005500.json
17:27:37 INFO skip cached page_006000.json
17:27:37 INFO skip cached page_006500.json
17:27:37 INFO skip cached page_007000.json
17:27:37 INFO skip cached page_007500.json
17:27:37 INFO skip cached page_008000.json
17:27:37 INFO skip cached page_008500.json
17:27:37 INFO skip cached page_009000.json
17:27:37 INFO skip cached page_009500.json
17:27:37 INFO skip cached page_010000.json
17:27:37 INFO skip cached page_010500.json
17:27:37 

done. 46 pages, total=22729


## 2. Flatten to project-level dataframe

In [13]:
def flatten_sectors(s):
    """sector field is a list of {Name, Percent} dicts; return list of names."""
    if not s:
        return []
    if isinstance(s, list):
        return [d.get("Name") for d in s if isinstance(d, dict) and d.get("Name")]
    if isinstance(s, dict):
        return [s.get("Name")] if s.get("Name") else []
    return []

rows = []
for page_path in sorted(RAW_DIR.glob("page_*.json")):
    payload = json.loads(page_path.read_text())
    for pid, rec in (payload.get("projects") or {}).items():
        rows.append({
            "id": rec.get("id") or pid,
            "project_name": rec.get("project_name"),
            "country": rec.get("countryshortname"),
            "countrycode": rec.get("countrycode"),
            "region": rec.get("regionname"),
            "sectors": flatten_sectors(rec.get("sector")),
            "sector1": (rec.get("sector1") or {}).get("Name") if isinstance(rec.get("sector1"), dict) else rec.get("sector1"),
            "themes": flatten_sectors(rec.get("theme")),
            "theme1": (rec.get("theme1") or {}).get("Name") if isinstance(rec.get("theme1"), dict) else rec.get("theme1"),
            "boardapprovaldate": rec.get("boardapprovaldate"),
            "closingdate": rec.get("closingdate"),
            "status": rec.get("status"),
            "totalamt": rec.get("totalamt"),
            "grantamt": rec.get("grantamt"),
            "lendinginstr": rec.get("lendinginstr"),
            "lendprojectcost": rec.get("lendprojectcost"),
        })

wb = pd.DataFrame(rows).drop_duplicates(subset=["id"]).reset_index(drop=True)
wb["boardapprovaldate"] = pd.to_datetime(wb["boardapprovaldate"], errors="coerce", utc=True)
wb["approval_year"] = wb.boardapprovaldate.dt.year
for col in ["totalamt", "grantamt", "lendprojectcost"]:
    wb[col] = pd.to_numeric(wb[col], errors="coerce")

wb_path = INTERIM / "wb_panel.parquet"
wb.drop(columns=["sectors", "themes"]).assign(
    sectors=wb.sectors.apply(lambda v: "; ".join(v) if isinstance(v, list) else ""),
    themes=wb.themes.apply(lambda v: "; ".join(v) if isinstance(v, list) else ""),
).to_parquet(wb_path, index=False)
print(f"wrote {wb_path} ({len(wb)} rows)")
wb.head(3)

wrote /Users/latahviawilliams/Downloads/Big_Data_export/final_project/ecosoc/data/interim/wb_panel.parquet (22586 rows)


,id,project_name,country,countrycode,region,sectors,sector1,themes,theme1,boardapprovaldate,closingdate,status,totalamt,grantamt,lendinginstr,lendprojectcost,approval_year
0,P505244,"Boosting Green Finance, Investment and Trade i...",Rwanda,[RW],Eastern and Southern Africa,[],,[],!$!0,2024-12-20 00:00:00+00:00,12/20/2025 12:00:00 AM,Active,NaN,NaN,Development Policy Lending,NaN,2024.0
1,P176429,Chattogram Water Supply Improvement Project,Bangladesh,[BD],South Asia,[],,[],!$!0,2024-12-19 00:00:00+00:00,12/31/2030 12:00:00 AM,Active,NaN,NaN,Investment Project Financing,NaN,2024.0
2,P181587,Transforming Agri-food Systems in Morocco,Morocco,[MA],Middle East and North Africa,[],,[],!$!0,2024-12-19 00:00:00+00:00,12/31/2029 12:00:00 AM,Active,NaN,NaN,Program-for-Results Financing,NaN,2024.0


## 3. Aggregate to (sector, year) and (country, year)

In [14]:
exploded = wb.explode("sectors").rename(columns={"sectors": "sector"})
exploded = exploded[exploded.sector.notna() & exploded.approval_year.notna()]

sector_year = (exploded
               .groupby(["sector", "approval_year"], as_index=False)
               .agg(totalamt_sum=("totalamt", "sum"),
                    grantamt_sum=("grantamt", "sum"),
                    project_count=("id", "nunique")))
sector_year = sector_year.rename(columns={"approval_year": "year"})
sy_path = INTERIM / "wb_sector_year.parquet"
sector_year.to_parquet(sy_path, index=False)
print(f"wrote {sy_path} ({len(sector_year)} rows, {sector_year.sector.nunique()} sectors)")

country_year = (wb[wb.approval_year.notna()]
                .groupby(["country", "approval_year"], as_index=False)
                .agg(totalamt_sum=("totalamt", "sum"),
                     project_count=("id", "nunique")))
country_year = country_year.rename(columns={"approval_year": "year"})
cy_path = INTERIM / "wb_country_year.parquet"
country_year.to_parquet(cy_path, index=False)
print(f"wrote {cy_path} ({len(country_year)} rows, {country_year.country.nunique()} countries)")

wrote /Users/latahviawilliams/Downloads/Big_Data_export/final_project/ecosoc/data/interim/wb_sector_year.parquet (3375 rows, 148 sectors)
wrote /Users/latahviawilliams/Downloads/Big_Data_export/final_project/ecosoc/data/interim/wb_country_year.parquet (5474 rows, 203 countries)


## 4. Quick QA

In [15]:
print("projects:", len(wb))
print("approval years:", int(wb.approval_year.min()), "-", int(wb.approval_year.max()))
print("non-null totalamt:", int(wb.totalamt.notna().sum()))
print("sum totalamt (USD):", f"{wb.totalamt.sum():,.0f}")
print("\ntop 10 sectors by total commitment:")
print(sector_year.groupby('sector').totalamt_sum.sum().sort_values(ascending=False).head(10))

projects: 22586
approval years: 1947 - 2024
non-null totalamt: 3624
sum totalamt (USD): 0

top 10 sectors by total commitment:
sector
(Historic)Agricultural credit                        0.0
Micro- and SME finance                               0.0
Non-Renewable Energy Generation                      0.0
Oil and Gas                                          0.0
Other Agriculture, Fishing and Forestry              0.0
Other Education                                      0.0
Other Energy and Extractives                         0.0
Other Industry, Trade and Services                   0.0
Other Information and Communications Technologies    0.0
Other Non-bank Financial Institutions                0.0
Name: totalamt_sum, dtype: float64
